In [0]:
import os
from io import StringIO
import requests
import pandas as pd
from tqdm import tqdm

# 1. Caminho do seu Volume no Databricks
CAMINHO_VOLUME = "/Volumes/workspace/default/fontes_tcc"

# 2. URLs oficiais da API SIDRA/IBGE com /v/allxp e os filtros de Esgoto (c11558/46290) e Água (c301/31471)
FONTES_SIDRA = {
    "Populacao_Tabela4714": {
        "url": "https://apisidra.ibge.gov.br/values/t/4714/n6/all/v/93/p/2022",
        "arquivo": "IBGE_SIDRA_Tabela4714_Populacao.csv"
    },
    "Esgotamento_Sanitario_Tabela9397": {
        "url": "https://apisidra.ibge.gov.br/values/t/9397/n6/all/v/allxp/p/2022/c11558/46290",
        "arquivo": "IBGE_SIDRA_Tabela9397_Esgoto.csv"
    },
    "Abastecimento_Agua_Tabela6909": {
        "url": "https://apisidra.ibge.gov.br/values/t/6909/n6/all/v/allxp/p/2022/c301/31471",
        "arquivo": "IBGE_SIDRA_Tabela6909_Agua.csv"
    }
}

print("=" * 70)
print(" COLETANDO TABELAS DO CENSO 2022 DIRETO DO SITE SIDRA/IBGE")
print("=" * 70)

dfs_sidra = {}

for nome_base, info in FONTES_SIDRA.items():
    url = info["url"]
    caminho_salvar = os.path.join(CAMINHO_VOLUME, info["arquivo"])
    
    print(f"\nTentando acessar: {url}")
    resposta = requests.get(url, stream=True, timeout=120)
    
    if resposta.status_code == 200:
        print(f"URL encontrada: {url}")
        conteudo_bytes = b""
        
        with tqdm(desc=f"Baixando: {nome_base}", unit="B", unit_scale=True, unit_divisor=1024) as barra:
            for bloco in resposta.iter_content(chunk_size=8192):
                conteudo_bytes += bloco
                barra.update(len(bloco))
                
        texto_json = StringIO(conteudo_bytes.decode("utf-8"))
        dados_json = pd.read_json(texto_json)
        
        cabecalho = dados_json.iloc[0]
        df_temp = dados_json[1:].copy()
        df_temp.columns = cabecalho
        
        df_temp.to_csv(caminho_salvar, index=False, sep=";", encoding="utf-8-sig")
        dfs_sidra[nome_base] = df_temp
        
        print(f"[SUCESSO] {len(df_temp)} municípios extraídos e salvos em: {caminho_salvar}")
    else:
        print(f"[AVISO] Status HTTP {resposta.status_code} para {nome_base} -> {resposta.text[:200]}")

# Compara os valores das 3 tabelas para os primeiros municípios
print("\n--- CONFERÊNCIA DOS VALORES EXTRAÍDOS (POPULAÇÃO vs. ESGOTO vs. ÁGUA) ---")
for nome_base, df in dfs_sidra.items():
    print(f"\n>> {nome_base}:")
    print(df[["Município (Código)", "Município", "Valor"]].head(3))

 COLETANDO TABELAS DO CENSO 2022 DIRETO DO SITE SIDRA/IBGE

Tentando acessar: https://apisidra.ibge.gov.br/values/t/4714/n6/all/v/93/p/2022
URL encontrada: https://apisidra.ibge.gov.br/values/t/4714/n6/all/v/93/p/2022


Baixando: Populacao_Tabela4714: 1.38MB [00:00, 2.55MB/s]


[SUCESSO] 5570 municípios extraídos e salvos em: /Volumes/workspace/default/fontes_tcc/IBGE_SIDRA_Tabela4714_Populacao.csv

Tentando acessar: https://apisidra.ibge.gov.br/values/t/9397/n6/all/v/allxp/p/2022/c11558/46290
URL encontrada: https://apisidra.ibge.gov.br/values/t/9397/n6/all/v/allxp/p/2022/c11558/46290


Baixando: Esgotamento_Sanitario_Tabela9397: 2.70MB [00:00, 4.13MB/s]


[SUCESSO] 5570 municípios extraídos e salvos em: /Volumes/workspace/default/fontes_tcc/IBGE_SIDRA_Tabela9397_Esgoto.csv

Tentando acessar: https://apisidra.ibge.gov.br/values/t/6909/n6/all/v/allxp/p/2022/c301/31471
URL encontrada: https://apisidra.ibge.gov.br/values/t/6909/n6/all/v/allxp/p/2022/c301/31471


Baixando: Abastecimento_Agua_Tabela6909: 2.60MB [00:00, 5.69MB/s]


[SUCESSO] 5570 municípios extraídos e salvos em: /Volumes/workspace/default/fontes_tcc/IBGE_SIDRA_Tabela6909_Agua.csv

--- CONFERÊNCIA DOS VALORES EXTRAÍDOS (POPULAÇÃO vs. ESGOTO vs. ÁGUA) ---

>> Populacao_Tabela4714:
0 Município (Código)                   Município  Valor
1            1100015  Alta Floresta D'Oeste - RO  21494
2            1100023              Ariquemes - RO  96833
3            1100031                 Cabixi - RO   5351

>> Esgotamento_Sanitario_Tabela9397:
0 Município (Código)                   Município Valor
1            1100015  Alta Floresta D'Oeste - RO   101
2            1100023              Ariquemes - RO  2034
3            1100031                 Cabixi - RO    74

>> Abastecimento_Agua_Tabela6909:
0 Município (Código)                   Município  Valor
1            1100015  Alta Floresta D'Oeste - RO   5342
2            1100023              Ariquemes - RO  52225
3            1100031                 Cabixi - RO   2016


In [0]:
# Célula 2: Verificar as colunas e os primeiros valores das 3 tabelas baixadas da API do SIDRA
for nome, df in dfs_sidra.items():
    print(f"\n=== {nome} ===")
    print("Colunas:", df.columns.tolist())
    print(df[["Município (Código)", "Município", "Variável", "Valor"]].head(3))


=== Populacao_Tabela4714 ===
Colunas: ['Nível Territorial (Código)', 'Nível Territorial', 'Unidade de Medida (Código)', 'Unidade de Medida', 'Valor', 'Município (Código)', 'Município', 'Variável (Código)', 'Variável', 'Ano (Código)', 'Ano']
0 Município (Código)                   Município             Variável  Valor
1            1100015  Alta Floresta D'Oeste - RO  População residente  21494
2            1100023              Ariquemes - RO  População residente  96833
3            1100031                 Cabixi - RO  População residente   5351

=== Esgotamento_Sanitario_Tabela9397 ===
Colunas: ['Nível Territorial (Código)', 'Nível Territorial', 'Unidade de Medida (Código)', 'Unidade de Medida', 'Valor', 'Município (Código)', 'Município', 'Variável (Código)', 'Variável', 'Ano (Código)', 'Ano', 'Tipo de esgotamento sanitário (Código)', 'Tipo de esgotamento sanitário', 'Grupo de idade (Código)', 'Grupo de idade', 'Cor ou raça (Código)', 'Cor ou raça', 'Existência de banheiro ou sanitário 

In [0]:
# Célula 3: Consultar no servidor do IBGE os códigos das categorias das Tabelas 9397 e 6909
import requests

for tabela in ["9397", "6909"]:
    url_desc = f"https://apisidra.ibge.gov.br/DescritoresTabela/t/{tabela}"
    resp = requests.get(url_desc).json()
    print(f"\n================ TABELA {tabela}: {resp.get('Nome')} ================")
    for classif in resp.get("Classificacoes", []):
        print(f"\nClassificação c{classif['Id']}: {classif['Nome']}")
        for cat in classif.get("Categorias", [])[:10]:
            print(f"   -> Código {cat['Id']}: {cat['Nome']}")


================ TABELA 9397: Moradores em domicílios particulares permanentes ocupados, por existência de banheiro ou sanitário e número de banheiros de uso exclusivo do domicílio e por tipo de esgotamento sanitário, segundo grupos de idade e cor ou raça ================

Classificação c458: Existência de banheiro ou sanitário e número de banheiros de uso exclusivo do domicílio
   -> Código 72117: Total
   -> Código 12032: Tinham banheiro de uso exclusivo do domicílio
   -> Código 12033: Tinham banheiro de uso exclusivo do domicílio - 1 banheiro
   -> Código 12034: Tinham banheiro de uso exclusivo do domicílio - 2 banheiros
   -> Código 12035: Tinham banheiro de uso exclusivo do domicílio - 3 banheiros
   -> Código 12036: Tinham banheiro de uso exclusivo do domicílio - 4 banheiros ou mais
   -> Código 72118: Apenas banheiro de uso comum a mais de um domicílio
   -> Código 72119: Apenas sanitário ou buraco para dejeções, inclusive os localizados no terreno
   -> Código 12046: Não tinh